In [1]:
import pandas as pd
import numpy as np
import os

def backtest_csv_signals(df, 
                         test_start_date='2020-01-31', 
                         test_end_date='2024-11-30',
                         buys_path_template='buys_{}.csv',
                         sells_path_template='sells_{}.csv',
                         transaction_cost=0.001,
                         verbose=True):
    """
    Backtest Long-Short portfolios directly from Buy/Sell CSV lists.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Must contain: 'datadate', 'permno', 'ret_fwd_1', 'mve_m'
    test_start_date, test_end_date : str
        Backtest range.
    buys/sells_path_template : str
        Path with {} placeholder for year (e.g., 'data/buys_{}.csv').
    """
    
    # 1. Setup Data
    df = df.copy()
    df['datadate'] = pd.to_datetime(df['datadate'])
    
    all_dates = sorted(df['datadate'].unique())
    start_dt = pd.to_datetime(test_start_date)
    end_dt = pd.to_datetime(test_end_date)
    
    # Filter dates to test range
    test_dates = [d for d in all_dates if d >= start_dt and d <= end_dt]
    
    # Storage
    results = {
        'ew': {'ret': [], 'gross': [], 'to': [], 'date': [], 'w_long': [], 'w_short': []},
        'vw': {'ret': [], 'gross': [], 'to': [], 'date': [], 'w_long': [], 'w_short': []}
    }
    
    # State tracking for turnover (Previous Weights & Returns)
    # Structure: {'ew': {'weights': {}, 'returns': {}, 'gross_ret': 0.0}, ...}
    state = {
        'ew': {'w': {}, 'r': {}, 'g': 0.0},
        'vw': {'w': {}, 'r': {}, 'g': 0.0}
    }
    
    # Cache for CSV signals to avoid re-reading disk
    signal_cache = {}

    if verbose:
        print("="*80)
        print("STARTING DIRECT CSV SIGNAL BACKTEST")
        print("="*80)

    # 2. Main Time Loop
    for i, current_date in enumerate(test_dates):
        current_year = current_date.year
        date_str = current_date.strftime('%Y-%m-%d')
        
        # --- A. Load Signals (Yearly) ---
        if current_year not in signal_cache:
            try:
                b_df = pd.read_csv(buys_path_template.format(current_year))
                s_df = pd.read_csv(sells_path_template.format(current_year))
                
                # Assuming CSVs have a 'permno' column. 
                # If permno is the index in your CSV, use b_df.index
                # Adjust column name 'permno' if your CSV header is different
                buys = set(b_df['permno'].values) if 'permno' in b_df.columns else set(b_df.index)
                sells = set(s_df['permno'].values) if 'permno' in s_df.columns else set(s_df.index)
                
                # Ensure no overlap (Long vs Short)
                common = buys.intersection(sells)
                if common:
                    if verbose: print(f"  ⚠ Warning: {len(common)} permnos in both lists. Removing from both.")
                    buys = buys - common
                    sells = sells - common
                    
                signal_cache[current_year] = {'longs': buys, 'shorts': sells}
                
            except FileNotFoundError:
                if verbose: print(f"  ⚠ Files for {current_year} not found. No positions.")
                signal_cache[current_year] = {'longs': set(), 'shorts': set()}

        long_candidates = signal_cache[current_year]['longs']
        short_candidates = signal_cache[current_year]['shorts']

        # --- B. Get Current Market Data ---
        # Filter df for current date and relevant permnos
        day_data = df[df['datadate'] == current_date].set_index('permno')
        
        # Valid assets must have return data
        valid_longs = [p for p in long_candidates if p in day_data.index and pd.notna(day_data.loc[p, 'ret_fwd_1'])]
        valid_shorts = [p for p in short_candidates if p in day_data.index and pd.notna(day_data.loc[p, 'ret_fwd_1'])]
        
        # Get returns dictionary for calculations
        current_returns = day_data['ret_fwd_1'].to_dict() # Includes all available permnos this day

        # If no valid signals, liquidate everything
        if not valid_longs and not valid_shorts:
            if verbose: print(f"[{i+1}] {date_str}: No valid signals. Liquidating.")
            _handle_liquidation(results, state, current_returns, transaction_cost, current_date)
            continue

        # --- C. Construct Weights ---
        
        # 1. Equal-Weighted (EW)
        ew_w = {}
        # Long side (+1.0 total)
        if valid_longs:
            w_l = 1.0 / len(valid_longs)
            for p in valid_longs: ew_w[p] = w_l
        # Short side (-1.0 total)
        if valid_shorts:
            w_s = -1.0 / len(valid_shorts)
            for p in valid_shorts: ew_w[p] = w_s

        # 2. Value-Weighted (VW)
        vw_w = {}
        # Get MVE for valid assets
        l_mve = {p: day_data.loc[p, 'mve_m'] for p in valid_longs if pd.notna(day_data.loc[p, 'mve_m'])}
        s_mve = {p: day_data.loc[p, 'mve_m'] for p in valid_shorts if pd.notna(day_data.loc[p, 'mve_m'])}
        
        # Normalize Longs
        tot_l = sum(l_mve.values())
        if tot_l > 0:
            for p, m in l_mve.items(): vw_w[p] = m / tot_l
        # Normalize Shorts
        tot_s = sum(s_mve.values())
        if tot_s > 0:
            for p, m in s_mve.items(): vw_w[p] = -m / tot_s # Negative weight for shorts

        # --- D. Execute & Calculate Returns ---
        _process_strategy('ew', ew_w, current_returns, state, results, transaction_cost, current_date)
        _process_strategy('vw', vw_w, current_returns, state, results, transaction_cost, current_date)
        
        if verbose:
            print(f"[{i+1}] {date_str} | L: {len(valid_longs)} S: {len(valid_shorts)} | "
                  f"EW: {results['ew']['ret'][-1]:.2%} | VW: {results['vw']['ret'][-1]:.2%}")

    # 3. Compile Output
    df_ew = pd.DataFrame(results['ew']).set_index('date')
    df_vw = pd.DataFrame(results['vw']).set_index('date')
    
    df_ew['cum_ret'] = (1 + df_ew['ret']).cumprod() - 1
    df_vw['cum_ret'] = (1 + df_vw['ret']).cumprod() - 1

    return df_ew, df_vw

# --- Helper Functions ---

def _handle_liquidation(results, state, current_returns, tc, date):
    """Liquidates portfolios if signals dry up."""
    for strat in ['ew', 'vw']:
        prev_w = state[strat]['w']
        prev_r = state[strat]['r']
        prev_g = state[strat]['g']
        
        if not prev_w:
            turnover, cost = 0.0, 0.0
        else:
            # Drift previous weights
            drifted = {}
            for p, w in prev_w.items():
                r = prev_r.get(p, 0.0)
                drifted[p] = w * (1 + r) / (1 + prev_g) if (1+prev_g) != 0 else 0
            
            # Liquidate means moving from Drifted -> 0
            turnover = sum(abs(w) for w in drifted.values())
            cost = turnover * tc
            
        results[strat]['ret'].append(-cost)
        results[strat]['gross'].append(0.0)
        results[strat]['to'].append(turnover)
        results[strat]['date'].append(date)
        results[strat]['w_long'].append({})
        results[strat]['w_short'].append({})
        
        # Reset State
        state[strat] = {'w': {}, 'r': {}, 'g': 0.0}

def _process_strategy(strat, new_w, current_returns, state, results, tc, date):
    """Calculates returns, turnover, and updates state."""
    prev_w = state[strat]['w']
    prev_r = state[strat]['r']
    prev_g = state[strat]['g']
    
    # 1. Calculate Turnover
    if not prev_w:
        # First period: turnover is just the sum of new weights (usually 2.0 for L/S)
        turnover = sum(abs(w) for w in new_w.values())
    else:
        # Calculate drifted weights from previous period
        drifted = {}
        all_assets = set(prev_w.keys()).union(new_w.keys())
        
        denom = (1 + prev_g)
        for p in all_assets:
            w_old = prev_w.get(p, 0.0)
            if w_old != 0:
                r_old = prev_r.get(p, 0.0)
                drifted[p] = w_old * (1 + r_old) / denom
            else:
                drifted[p] = 0.0
                
        # Turnover = |New - Drifted|
        turnover = sum(abs(new_w.get(p, 0.0) - drifted.get(p, 0.0)) for p in all_assets)

    # 2. Gross Return (Current weights * Current returns)
    gross = sum(w * current_returns.get(p, 0.0) for p, w in new_w.items())
    
    # 3. Transaction Cost & Net Return
    # Cost is paid on the rebalancing volume based on portfolio value at end of period? 
    # Standard approximation: Cost * Turnover * (1 + Gross) or just Cost * Turnover
    # Using your previous logic: Cost * (1 + Gross) * Turnover
    cost = tc * (1 + gross) * turnover
    net = gross - cost
    
    # 4. Store Results
    results[strat]['ret'].append(net)
    results[strat]['gross'].append(gross)
    results[strat]['to'].append(turnover)
    results[strat]['date'].append(date)
    
    longs = {p: w for p, w in new_w.items() if w > 0}
    shorts = {p: w for p, w in new_w.items() if w < 0}
    results[strat]['w_long'].append(longs)
    results[strat]['w_short'].append(shorts)
    
    # 5. Update State for next period
    state[strat]['w'] = new_w
    state[strat]['r'] = current_returns # Store returns to calculate drift next time
    state[strat]['g'] = gross

In [3]:
# Run the backtest

df = pd.read_csv('../../green cleaned.csv', dtype={'ncusip': 'string'})
df['ret_fwd_1'] = df.groupby('permno')['ret_excess'].shift(-1)

df_ew, df_vw = backtest_csv_signals(
    df=df,
    test_start_date='2020-01-31',
    test_end_date='2024-11-30',
    buys_path_template='buys_{}.csv',   # Ensure these match your filenames
    sells_path_template='sells_{}.csv',
    transaction_cost=0.001
)

# Print Summary Metrics
print("Equal Weighted Total Return:", df_ew['cum_ret'].iloc[-1])
print("Value Weighted Total Return:", df_vw['cum_ret'].iloc[-1])

print("EW Sharpe:", df_ew['ret'].mean() / df_ew['ret'].std() * np.sqrt(12))
print("VW Sharpe:", df_vw['ret'].mean() / df_vw['ret'].std() * np.sqrt(12))

STARTING DIRECT CSV SIGNAL BACKTEST
[1] 2020-01-31 | L: 29 S: 11 | EW: -2.96% | VW: -3.58%
[2] 2020-02-29 | L: 29 S: 11 | EW: -5.80% | VW: -8.35%
[3] 2020-03-31 | L: 29 S: 11 | EW: -0.43% | VW: -1.88%
[4] 2020-04-30 | L: 28 S: 11 | EW: -2.79% | VW: -5.32%
[5] 2020-05-31 | L: 28 S: 10 | EW: 0.77% | VW: -1.08%
[6] 2020-06-30 | L: 28 S: 10 | EW: -9.51% | VW: -8.67%
[7] 2020-07-31 | L: 28 S: 10 | EW: -6.04% | VW: -2.23%
[8] 2020-08-31 | L: 28 S: 9 | EW: 0.51% | VW: -0.18%
[9] 2020-09-30 | L: 28 S: 9 | EW: 3.10% | VW: 2.84%
[10] 2020-10-31 | L: 28 S: 9 | EW: 3.67% | VW: 6.96%
[11] 2020-11-30 | L: 28 S: 9 | EW: -1.79% | VW: 0.26%
[12] 2020-12-31 | L: 28 S: 9 | EW: -1.29% | VW: -0.58%
[13] 2021-01-31 | L: 3 S: 118 | EW: -12.10% | VW: -9.80%
[14] 2021-02-28 | L: 3 S: 115 | EW: 6.85% | VW: 5.15%
[15] 2021-03-31 | L: 3 S: 115 | EW: -0.89% | VW: -0.59%
[16] 2021-04-30 | L: 3 S: 114 | EW: -1.39% | VW: 2.20%
[17] 2021-05-31 | L: 3 S: 113 | EW: -5.04% | VW: -7.05%
[18] 2021-06-30 | L: 3 S: 113 | EW:

In [5]:
df_ew, df_vw = backtest_csv_signals(
    df=df,
    test_start_date='2015-01-31',
    test_end_date='2024-11-30',
    buys_path_template='buys_{}.csv',   # Ensure these match your filenames
    sells_path_template='sells_{}.csv',
    transaction_cost=0.001
)

# Print Summary Metrics
print("Equal Weighted Total Return:", df_ew['cum_ret'].iloc[-1])
print("Value Weighted Total Return:", df_vw['cum_ret'].iloc[-1])

print("EW Sharpe:", df_ew['ret'].mean() / df_ew['ret'].std() * np.sqrt(12))
print("VW Sharpe:", df_vw['ret'].mean() / df_vw['ret'].std() * np.sqrt(12))

STARTING DIRECT CSV SIGNAL BACKTEST
[1] 2015-01-31 | L: 35 S: 19 | EW: -4.54% | VW: -1.38%
[2] 2015-02-28 | L: 34 S: 17 | EW: -0.48% | VW: 0.25%
[3] 2015-03-31 | L: 34 S: 17 | EW: -0.16% | VW: 2.97%
[4] 2015-04-30 | L: 34 S: 17 | EW: 4.78% | VW: 4.33%
[5] 2015-05-31 | L: 33 S: 17 | EW: -2.62% | VW: -0.06%
[6] 2015-06-30 | L: 33 S: 17 | EW: 5.09% | VW: 4.83%
[7] 2015-07-31 | L: 33 S: 17 | EW: 0.30% | VW: -0.51%
[8] 2015-08-31 | L: 33 S: 17 | EW: 5.45% | VW: 2.15%
[9] 2015-09-30 | L: 33 S: 17 | EW: 0.76% | VW: 3.42%
[10] 2015-10-31 | L: 33 S: 17 | EW: 2.54% | VW: 2.54%
[11] 2015-11-30 | L: 33 S: 17 | EW: -0.50% | VW: -2.22%
[12] 2015-12-31 | L: 33 S: 16 | EW: -4.80% | VW: -5.85%
[13] 2016-01-31 | L: 41 S: 38 | EW: -2.22% | VW: -3.52%
[14] 2016-02-29 | L: 41 S: 35 | EW: 2.10% | VW: 0.84%
[15] 2016-03-31 | L: 41 S: 34 | EW: 5.64% | VW: 4.40%
[16] 2016-04-30 | L: 41 S: 34 | EW: -3.36% | VW: -3.99%
[17] 2016-05-31 | L: 41 S: 34 | EW: -1.97% | VW: -2.06%
[18] 2016-06-30 | L: 41 S: 34 | EW: -3